# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Ahmed-Zia/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 2: Refresh / Content Opportunity Scoring.

Task type: primarily a SCORING / RANKING task, built on top of a
binary classification model underneath. The end deliverable per the
lane guide is "a ranked review queue with scores, actions, and reason
codes" — not just a yes/no label. Concretely: I train a classifier to
estimate P(page needs review), then rank all pages by that probability
to produce the queue. This is the same shape as the starter pipeline:
scripts/03_train_model.py outputs a probability, and
04_evaluate_and_export.py turns that into a ranked, reason-coded queue.

In [3]:
task_type = "Scoring/Ranking (binary classification probability used as the ranking score)"
print(f"Task type: {task_type}")

Task type: Scoring/Ranking (binary classification probability used as the ranking score)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Starter proxy target: is_declining_label = (trend_direction == "down")

This is what the starter pipeline actually predicts, and it's what
I'll use for this week's framing exercise since it's what's available
in the starter CSV. I'm naming its weakness up front, per the lane
guide (Section 5): this is a CURRENT-window bucket, not a future
outcome, so it's a beginner proxy label, not the ideal capstone
target. A stronger version — which I'll move toward once I pull the
warehouse release — would be a future-window label:

  features from prior 90 days -> decline over next 30 days

That version avoids using information that's only knowable "now" to
predict something that's also true "now."

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Muhammad-Ahmed-Zia/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Still not found — check the repo URL/branch"

Working dir: /flyrank-ml-internship


In [5]:
import pandas as pd, os

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Target: is_declining_label")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Target: is_declining_label
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: Precision@50.

Per the lane guide (Section 11), top-K metrics beat generic accuracy
here because they match how the output is actually used: a review
team with limited weekly capacity works through a fixed-size queue,
not the whole dataset. Precision@50 asks "of the top 50 pages we say
to review first, how many are actually right?" — directly matching a
team that can act on ~50 candidates a week.

Secondary metrics I'll track alongside it: average precision (rewards
getting the whole ranking right, not just the top slice) and recall
at that same K, since Section 2's framing established that false
negatives (missed decliners) are costlier than false positives given
capacity constraints — a metric like Precision@50 alone won't catch
that trade-off, so I'll watch recall too.

In [6]:
import json

res = json.load(open("outputs/model_results.json")) if os.path.exists("outputs/model_results.json") else None
if res:
    print("Baseline  Precision@50:", res["baseline"]["baseline_precision_at_50"])
    print("RF model  Precision@50:", res["models"]["random_forest"]["precision_at_50"])
else:
    print("Reference values from docs/ml-intern-dataset-and-lane-guide.md Section 5:")
    print("baseline rules   Precision@50: 0.240")
    print("random forest    Precision@50: 0.740")


Reference values from docs/ml-intern-dataset-and-lane-guide.md Section 5:
baseline rules   Precision@50: 0.240
random forest    Precision@50: 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (content_id), scored at a single point in
time. Not a client, not a query, not a day — a page. Below is the
actual dataframe sliced to the columns this lane cares about, so the
grain is visible, not just claimed.

In [7]:
lane2_cols = [
    "content_id", "client_id", "impressions_90d", "sessions_90d",
    "avg_position", "ctr", "days_since_last_update", "content_age_days",
    "word_count", "content_type", "trend_direction", "is_declining_label"
]

lane2_view = df[lane2_cols].copy()
print("Shape (rows, cols):", lane2_view.shape)
print("One row = one content_id, at this snapshot in time\n")
lane2_view.head(5)


Shape (rows, cols): (30000, 12)
One row = one content_id, at this snapshot in time



,content_id,client_id,impressions_90d,sessions_90d,avg_position,ctr,days_since_last_update,content_age_days,word_count,content_type,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,10.6,0.76,20,187,3221.0,keyword article,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,20.3,0.05,25,445,2481.0,keyword article,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,20,141,3515.0,keyword article,down,1
3,content_331d6c4de07b,client_19581e27de,11751,78,6.2,0.49,22,463,NaN,keyword article,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,44.0,0.13,14,263,2803.0,keyword article,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Two pieces of evidence, both already computed:

1. Weak-obvious-signal check (Week 1 discovery): search_volume barely
   correlates with actual impressions_90d (r ~ 0.001), and declining
   vs growing pages have nearly identical median word counts. Simple,
   "obvious" rules based on volume or length would misallocate a
   reviewer's limited time.

2. Direct comparison on this exact task: the starter's hand-written
   rule (0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity
   + 0.05*depth_gap) scores 0.240 on Precision@50. A random forest
   trained on the same observable signals scores 0.740 — roughly 3x
   better, validated with client-holdout splitting so no client's
   pages leaked between train and test.

A fixed rule can only combine a few signals in one hard-coded way.
The data has enough real, non-obvious structure (per point 1) that a
model finds interaction effects a human-written formula misses — and
point 2 shows that gap is large, not marginal. That's the case for ML
over a rule here specifically, not as a general "ML is better" claim.

In [8]:
# Sketch: how the fixed baseline and a learned target compare, structurally
baseline_formula = "0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap"
print("Fixed rule (hand-coded weights):", baseline_formula)
print("Learned model: same underlying signals, weights AND interactions learned from data")
print(f"\nObserved gap on this task: baseline 0.240 -> model 0.740 Precision@50 (~3.1x)")

Fixed rule (hand-coded weights): 0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap
Learned model: same underlying signals, weights AND interactions learned from data

Observed gap on this task: baseline 0.240 -> model 0.740 Precision@50 (~3.1x)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- Task type named: yes — scoring/ranking, built on a binary
  classification probability.
- Target/proxy named, with its weakness flagged: yes —
  is_declining_label (current-window proxy), future-window version
  noted as the stronger capstone direction.
- Success metric named and justified: yes — Precision@50, matched to
  real review capacity, with average precision and recall as
  secondary checks tied to the cost asymmetry from ML-02.
- Unit of analysis shown as a real dataframe: yes — one row = one
  content_id at a snapshot, printed above with actual data.
- Why ML beats a fixed rule, with evidence not opinion: yes — weak
  obvious-signal correlations + a measured 3.1x Precision@50 gap on
  this exact task.
- Tied to a real content action: yes — carries forward from ML-02,
  top-K queue feeds a weekly refresh sprint.